In [ ]:
# Set up an Azure Machine Learning workspace
from azureml.core import Workspace

# Create or retrieve an existing Azure ML workspace
ws = Workspace.get(name='myworkspace',
                      subscription_id='your-subscription-id',
                      resource_group='myresourcegroup',
                      location='eastus')

ws = Workspace.create(name='myworkspace',
                      subscription_id='your-subscription-id',
                      resource_group='myresourcegroup',
                      location='eastus')

# Write configuration to the workspace config file
ws.write_config(path='.azureml')

In [9]:
import torch 
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets,transforms
# Define a simple neural netwrok with one hidden layer
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()  # Activation function
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Initialize the neural networl
model = SimpleNN()

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss() # For Classification tasks
optimizer = optim.SGD(model.parameters(), lr = 0.01) # Stochastic Gradient Descent

# Load MNIST dataset
train_loader = torch.utils.data.DataLoader(
    datasets.MNIST(root ='data', train = True, download = True, 
                  transform = transforms.ToTensor()),
    batch_size = 32, shuffle = True)

# Train model
num_epochs = 5
for epoch in range(num_epochs):
    running_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad() # Reset gradients
        # Forward pass 
        outputs = model(inputs.view(-1, 784)) #Flatten input
        loss = criterion(outputs, labels)

        # Backward pass and optimization 
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}')

Epoch 1, Loss: 0.8814804394721985
Epoch 2, Loss: 0.3768697164972623
Epoch 3, Loss: 0.32224275259375573
Epoch 4, Loss: 0.2933862342854341
Epoch 5, Loss: 0.2718213941474756


In [10]:
# model deployment
from azureml.core import Model
# Save the trained model
torch.save(model.state_dict(), 'simple_nn.pth')

# Register the model in Azure
model = Model.register(workspace = ws, model_path = 'simple_nn.pth', model_name='simple_nn')

ModuleNotFoundError: No module named 'azureml'